# AI & Student Learning
**INFO4125 · Team 2**

Explore associations between AI use and academic outcomes. This notebook is the reproducible analysis behind the dashboard. Restart the kernel and run all cells in order.

Research questions:
1. How do GPA change and skill retention vary with weekly AI hours?
2. Do outcomes differ by AI use case or academic background?
3. How do burnout categories vary with reported AI dependency?

The results describe this dataset. They do not establish causal effects, population estimates, or safe AI-use thresholds.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "analysis.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the project repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.io as pio
from IPython.display import display, Markdown
from src.analysis import *
from src.charts import *
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_columns", 20)
raw = load_data()
print(f"{len(raw):,} records × {len(raw.columns)} source columns")
raw.head()

## 1. Source and definitions
[Dataset and column descriptions](https://www.kaggle.com/datasets/laveshjadon/ai-impact-on-students). Source metadata lists CC0. [Exploration notebook that inspired the topic coverage](https://www.kaggle.com/code/laveshjadon/data-exploration-and-cleaning-ipynb); code here is original.

**Provenance remains unverified:** recruitment, geography, collection dates, and measurement methods are undocumented. We cannot verify whether these are real or synthetic records. Age, country, and standardized-test scores are absent. Year of study includes graduates.

GPA is on a 0–4 scale. Dependency and exam anxiety use 1–10 scales. Skill retention uses 0–100. Burnout is a supplied categorical label, not a verified clinical measure. The CSV policy label `Strict_Ban` differs from `Strictly_Ban` in the source description; preserve the CSV value.

In [ ]:
dictionary = pd.DataFrame([
    {"Column": c, "Type": str(raw[c].dtype), "Missing": int(raw[c].isna().sum()), "Distinct": int(raw[c].nunique())}
    for c in raw.columns
])
display(dictionary)
display(raw.describe().T.round(3))

## 2. Validate before preparing
Check completeness, duplicates, categories, score ranges, and integer scales. Traditional study hours use a physical bound of 168 hours/week; observed extremes are not deleted. A review failure stops preparation rather than silently altering records.

In [ ]:
validation = validate_data(raw)
display(validation)
assert validation.Issues.sum() == 0, "Review validation issues before proceeding."
df = prepare_data(raw)
assert len(df) == len(raw)
assert "GPA_Change" not in raw.columns
print("Original rows retained; no values imputed. GPA change and boolean subscription prepared.")

## 3. Understand the sample
These distributions describe representation in the file, not representativeness of a real-world population.

In [ ]:
for column in ["Major_Category", "Year_of_Study", "Primary_Use_Case", "Burnout_Risk_Level"]:
    counts = df[column].value_counts().rename("Records").to_frame()
    counts["Percent"] = (100 * counts.Records / len(df)).round(2)
    display(Markdown(f"**{friendly(column)}**"))
    display(counts)
style(px.histogram(df, x="Weekly_GenAI_Hours", nbins=40, labels=LABELS)).show()

## 4. GPA change
`GPA_Change = Post_Semester_GPA − Pre_Semester_GPA`. Positive is improvement; negative is decline. A smaller positive change is still improvement. Before/after semester does not mean before/after AI adoption.

In [ ]:
display(df[["Pre_Semester_GPA", "Post_Semester_GPA", "GPA_Change"]].describe().T.round(3))
gpa_distribution(df).show()

## 5. AI hours and outcomes
Pearson measures linear association and Spearman measures rank association. A small overall correlation does not exclude a nonlinear relationship or subgroup differences. Five-hour bins below are fixed display choices, not discovered thresholds.

In [ ]:
correlations = correlation_table(df)
display(correlations.round(4))
for outcome in ["GPA_Change", "Skill_Retention_Score"]:
    display(Markdown(f"### {friendly(outcome)}"))
    hours_plot(df, outcome).show()
    display(hours_summary(df, outcome).round(3))

In [ ]:
pearson = correlations[correlations.Method == "Pearson"].set_index("Outcome").Correlation
display(Markdown(
    f"In this file, weekly AI hours have Pearson correlations of "
    f"**{pearson[friendly('GPA_Change')]:.3f}** with GPA change and "
    f"**{pearson[friendly('Skill_Retention_Score')]:.3f}** with skill retention. "
    "These are weak overall linear associations, not evidence that changing AI hours causes either outcome."
))
relationship_plot(df, "GPA_Change").show()
print("Scatterplot displays a fixed sample of at most 3,000; all summaries use the full dataset.")

## 6. How AI is used
Compare use cases without assuming one is better. Means are unadjusted, so differences in prior GPA, major, and study habits could explain apparent differences. Report group sizes, medians, and spread alongside means.

In [ ]:
for outcome in ["GPA_Change", "Skill_Retention_Score"]:
    comparison_plot(df, "Primary_Use_Case", outcome).show()
    display(group_summary(df, "Primary_Use_Case", outcome).round(3))

## 7. Academic background
These comparisons help identify follow-up questions. They do not isolate effects of major, year, or policy.

In [ ]:
comparison_plot(df, "Year_of_Study", "GPA_Change").show()
display(group_summary(df, "Major_Category", "GPA_Change").round(3))
display(group_summary(df, "Prompt_Engineering_Skill", "Skill_Retention_Score").round(3))

## 8. Dependency and burnout
Compare proportions within groups rather than raw counts across unequal groups. Burnout categories and dependency are dataset labels with insufficiently documented measurement methods.

In [ ]:
burnout_plot(df, "Perceived_AI_Dependency").show()
display(burnout_summary(df, "Perceived_AI_Dependency").round(2))
correlation_plot(df).show()

## 9. What can go into the dashboard?
- Overview of the selected records and source limitations.
- AI hours versus GPA change and retention, with counts.
- Use-case, major, and year comparisons.
- Burnout proportions by dependency or student group.

Filters describe subgroups; they do not simulate an intervention. Do not call five-hour band boundaries safe/harmful thresholds. The dashboard imports these same functions.

**Next research step:** verify provenance and measurement definitions. If justified, model post-semester GPA while accounting for baseline GPA and relevant covariates; this still requires caution about causal interpretation. Do not infer mechanisms from correlations.

## 10. Export derived analysis
This cell writes only to the ignored `outputs/` directory. The original CSV remains unchanged.

In [ ]:
import subprocess
subprocess.run([sys.executable, str(ROOT / "scripts" / "analyze.py")], cwd=ROOT, check=True)